In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import os
import numpy as np
import site

In [ ]:
reedspath = os.path.expanduser('~/github/ReEDS-2.0')
site.addsitedir(reedspath)
import reeds

In [ ]:
def detrend_dataframe(df):
    """
    For each column in a Pandas dataframe with a datetime index, fit a linear regression to the data,
    subtract the slope of the linear regression from the original data to remove the long-term trend,
    and normalize the data to the end of the datetime index range.

    Parameters:
    df (DataFrame): Pandas dataframe with a datetime index and series of columns.

    Returns:
    DataFrame: Detrended and normalized dataframe.
    """
    # # Create an empty dataframe to store detrended data
    # detrended_df = pd.DataFrame(index=df.index)

    # Convert datetime index to numeric for regression (using days since start)
    time_in_days = (df.index - df.index[0]).days
    X = time_in_days.values.reshape(-1, 1)

    # Iterate through each column in the dataframe
    detrended_data = {}
    for column in df.columns:
        # Fit linear regression
        y = df[column].values
        model = LinearRegression()
        model.fit(X, y)

        # Calculate trend
        trend = model.predict(X)

        # Detrend the data
        detrended = y + (trend[-1] - trend)

        # Add to the detrended dataframe
        # detrended_df_cols[column] = detrended
        detrended_data[column] = detrended

    detrended_df = pd.DataFrame(detrended_data, index=df.index)

    return detrended_df

In [ ]:
def roll_hourly_data(df_hr, source_timezone, output_timezone, load_source_hr_type):
    #If hour-beginning, we shift data down (+1) to convert to hour ending
    hour_end_shift = 1 if load_source_hr_type == 'begin' else 0
    #Extract the integer adjustment from UTC for source and output timezones
    source_tz_num = -1 * int(source_timezone.replace('Etc/GMT', ''))
    output_tz_num = -1 * int(output_timezone.replace('Etc/GMT', ''))
    #Shift timezone of hourly data
    shift = output_tz_num - source_tz_num + hour_end_shift
    if shift != 0:
        for ba in df_hr:
            df_hr[ba] = np.roll(df_hr[ba], shift)
    return df_hr

In [ ]:
## Detrend set of historic load spanning 2016-2023
read_location = '//nrelnas01/ReEDS/Supply_Curve_Data/LOAD/2025_Update/historic'
df_16_23 = pd.read_hdf(os.path.join(read_location, 'historic_load_hourly_2016_2023_ba.h5'))
df_16_23 = detrend_dataframe(df_16_23)

df_16_23_mean = df_16_23.mean(axis=0)

In [ ]:
## Detrend set of historic load spanning 2007-2013
read_location = '//nrelnas01/ReEDS/Supply_Curve_Data/LOAD/2020_Update/plexos_to_reeds/outputs'
df_hr = pd.read_csv(
    os.path.join(read_location, 'load_hourly_ba_EST.csv'),
    index_col='DATETIME',
    parse_dates=['DATETIME']
)
df_hr = df_hr[df_16_23.columns]

#Shift hourly data based on desired timezones. Shift each year independently.
load_source_timezone = "Etc/GMT+5"
output_timezone = "Etc/GMT+6"
df_hr_ls = []
for year in range(2007,2014):
    df_hr_yr = df_hr[df_hr.index.year == year].copy()
    df_hr_yr = roll_hourly_data(df_hr_yr, load_source_timezone, output_timezone, 'end')
    df_hr_ls.append(df_hr_yr)
df_hr = pd.concat(df_hr_ls)

df_07_13 = df_hr * df_16_23_mean / df_hr.mean(axis=0) # scale to new data
df_07_13.index = df_07_13.index.tz_localize('Etc/GMT+6')

In [ ]:
## Combine and clean the results
df = pd.concat([df_07_13, df_16_23]).round()
# Remove December 31 for leap years
df = df[~((df.index.year % 4 == 0) & (df.index.month == 12) & (df.index.day == 31))].copy()
df.index.names = ["datetime"]

In [ ]:
## Export
reeds.io.write_profile_to_h5(df, 'historic_load_hourly.h5', os.path.join(reedspath, 'inputs/load'))

In [ ]:
## Plot all load profiles, grouped by state
import matplotlib.pyplot as plt

df_hier = pd.read_csv(os.path.join(reedspath, "inputs/hierarchy.csv"))
def plot_timeseries_by_state(df):
    states = list(df_hier.loc[df_hier.country == 'USA']['st'].unique())
    for state in states:
        print(f"State: {state}")
        bas = list(df_hier.loc[df_hier.st == state]['ba'])
        if len(bas) <= 4:
            df[bas].plot(subplots=True, legend=False, title=bas)
            plt.tight_layout()
            plt.show()
        else:
            for ba in bas:
                df[ba].plot(legend=False, title=ba)
                plt.tight_layout()
                plt.show()

plot_timeseries_by_state(df)